# N-Body Gravitational Simulation

The **$N$-body problem** models the motion of $N$ particles under mutual gravitational (or generalised power-law) attraction. For particle $i$ with mass $m_i$ and position $x_i \in \mathbb{R}^2$:
$$
m_i \ddot{x}_i = \sum_{j \neq i} \frac{G m_i m_j (x_j - x_i)}{\|x_j - x_i\|^{1+\alpha}},
$$
where $\alpha \ge 1$ is the **force exponent** ($\alpha = 2$ for Newtonian gravity in 3D; $\alpha = 1$ in 2D).

## Verlet (leapfrog) integrator

The **Störmer–Verlet** (leapfrog) method is the workhorse for $N$-body simulation:
$$
v_i^{k+1/2} = v_i^{k-1/2} + h\, a_i^k, \qquad x_i^{k+1} = x_i^k + h\, v_i^{k+1/2},
$$
where $a_i^k = F_i(x^k)/m_i$. It is **symplectic** (preserves a modified Hamiltonian), yielding long-time stability that forward Euler lacks.

## Conservation laws

- **Total energy** $E = \sum_i \frac{1}{2}m_i\|v_i\|^2 - \sum_{i<j} \frac{G m_i m_j}{(\alpha-1)\|x_i-x_j\|^{\alpha-1}}$ is approximately conserved by Verlet.
- **Centre of mass**: $\sum m_i x_i$ and $\sum m_i v_i$ are exactly conserved.
- **Angular momentum**: $L = \sum_i m_i (x_i \times v_i)$ is conserved for spherical potentials.

## Softening

A **softening length** $\varepsilon > 0$ prevents divergence at close encounters:
$$
\|x_j - x_i\|^2 \to \|x_j - x_i\|^2 + \varepsilon^2.
$$

## Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

plt.rcParams['figure.dpi'] = 120

## N-body simulator

We implement the Verlet integrator with softening and a configurable force exponent.

In [ ]:
def compute_forces(pos, masses, G=1.0, alpha=2.0, eps=0.05):
    """Returns accelerations: a[i] = sum_{j!=i} G*m_j*(pos[j]-pos[i]) / dist^(alpha+1)"""
    N = len(masses)
    acc = np.zeros_like(pos)
    for i in range(N):
        for j in range(N):
            if i == j: continue
            r = pos[j] - pos[i]
            dist2 = np.dot(r, r) + eps**2
            dist_alpha = dist2**((alpha + 1) / 2)
            acc[i] += G * masses[j] * r / dist_alpha
    return acc

def simulate_nbody(pos0, vel0, masses, n_steps, h=0.05, G=1.0, alpha=2.0, eps=0.05):
    N = len(masses)
    pos = pos0.copy(); vel = vel0.copy()
    trajs = [pos.copy()]
    energies = []

    acc = compute_forces(pos, masses, G, alpha, eps)
    # half-step for leapfrog
    vel_half = vel + 0.5 * h * acc

    for _ in range(n_steps):
        pos = pos + h * vel_half
        acc = compute_forces(pos, masses, G, alpha, eps)
        vel_half = vel_half + h * acc
        trajs.append(pos.copy())
        vel_curr = vel_half - 0.5 * h * acc
        KE = 0.5 * np.sum(masses[:, None] * vel_curr**2)
        PE = 0.0
        for i in range(N):
            for j in range(i+1, N):
                d = np.linalg.norm(pos[i] - pos[j])
                PE -= G * masses[i] * masses[j] / d
        energies.append(KE + PE)

    return np.array(trajs), np.array(energies)

print('N-body simulator ready.')

## Three-body choreography

A classic figure-8 three-body solution: three equal-mass bodies chase each other along a figure-8 orbit.

In [ ]:
# Figure-8 initial conditions (Chenciner & Montgomery 2000)
masses3 = np.ones(3)
pos3 = np.array([[-0.97000436, 0.24308753],
                  [0.97000436, -0.24308753],
                  [0.0,        0.0]])
vel3 = np.array([[0.93240737/2, 0.86473146/2],
                  [0.93240737/2, 0.86473146/2],
                  [-0.93240737,  -0.86473146]])

trajs3, energies3 = simulate_nbody(pos3, vel3, masses3, n_steps=400, h=0.02, G=1.0, alpha=2.0, eps=1e-4)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cols3 = ['royalblue', 'tomato', 'seagreen']
for i, col in enumerate(cols3):
    axes[0].plot(trajs3[:,i,0], trajs3[:,i,1], '-', lw=1.5, color=col, label=f'body {i+1}')
    axes[0].plot(trajs3[0,i,0], trajs3[0,i,1], 'o', ms=8, color=col)
axes[0].set_aspect('equal'); axes[0].grid(alpha=0.2)
axes[0].set_title('Figure-8 three-body choreography'); axes[0].legend(fontsize=9)

axes[1].plot(energies3, 'k-', lw=1.5)
axes[1].set_xlabel('step'); axes[1].set_ylabel('total energy')
axes[1].set_title('Energy conservation (Verlet integrator)'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Random N-body system

We initialise $N$ particles at rest in a cold cluster and watch them collapse and virialise.

In [ ]:
rng = np.random.default_rng(42)
N_rand = 12
pos_r = rng.uniform(-1, 1, (N_rand, 2))
vel_r = 0.1 * rng.standard_normal((N_rand, 2))
masses_r = 0.5 + rng.uniform(0, 1, N_rand)

trajs_r, energies_r = simulate_nbody(pos_r, vel_r, masses_r, n_steps=300, h=0.03, G=1.0, alpha=2.0, eps=0.1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
cols_r = plt.cm.tab20(np.linspace(0, 1, N_rand))
for step, ax in zip([0, 100, 300], axes):
    for i in range(N_rand):
        tail = trajs_r[max(0,step-30):step+1, i, :]
        ax.plot(tail[:,0], tail[:,1], '-', lw=0.8, color=cols_r[i], alpha=0.6)
        ax.scatter(trajs_r[step,i,0], trajs_r[step,i,1],
                   s=40*masses_r[i], color=cols_r[i], edgecolors='k', lw=0.5, zorder=5)
    ax.set_aspect('equal'); ax.grid(alpha=0.2); ax.set_title(f't={step*0.03:.1f}')
    ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)
plt.tight_layout(); plt.show()

## Interactive: force exponent and time step

In [ ]:
def show_nbody(alpha=2.0, n_steps=200, eps=0.1):
    trajs_i, energies_i = simulate_nbody(pos_r, vel_r, masses_r, n_steps=n_steps,
                                          h=0.03, G=1.0, alpha=alpha, eps=eps)
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    for i in range(N_rand):
        axes[0].plot(trajs_i[:,i,0], trajs_i[:,i,1], '-', lw=0.8, color=cols_r[i], alpha=0.7)
        axes[0].scatter(trajs_i[-1,i,0], trajs_i[-1,i,1],
                        s=40*masses_r[i], color=cols_r[i], edgecolors='k', lw=0.5, zorder=5)
    axes[0].set_aspect('equal'); axes[0].grid(alpha=0.2)
    axes[0].set_title(f'$\\alpha={alpha}$, {n_steps} steps')
    axes[1].plot(energies_i, 'k-', lw=1.5)
    axes[1].set_xlabel('step'); axes[1].set_ylabel('energy')
    axes[1].set_title('Energy'); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

interact(show_nbody,
         alpha=FloatSlider(value=2.0, min=1.0, max=4.0, step=0.25, description='$\\alpha$'),
         n_steps=IntSlider(value=200, min=50, max=500, step=50, description='steps'),
         eps=FloatSlider(value=0.1, min=0.01, max=0.5, step=0.01, description='softening'));

## Bibliographical resources

- Chenciner, A. and Montgomery, R. (2000). A remarkable periodic solution of the three-body problem in the case of equal masses. *Annals of Mathematics*, 152(3), 881–901.
- Verlet, L. (1967). Computer experiments on classical fluids. I. Thermodynamical properties of Lennard–Jones molecules. *Physical Review*, 159(1), 98–103.
- Leimkuhler, B. and Reich, S. (2004). *Simulating Hamiltonian Dynamics*. Cambridge University Press.
- Aarseth, S. J. (2003). *Gravitational N-Body Simulations: Tools and Algorithms*. Cambridge University Press.
- Hut, P. and Makino, J. (1999). The art of computational science: The gravitational million-body problem. *Science*, 283(5406), 501–501.